# All-ligand distance-dependent GP tests for ATAC-derived modalities

This notebook reproduces the spatial Gaussian-process (GP) likelihood-ratio screens for **all 15 ligand perturbations** across four ATAC-derived feature spaces:

1. called ATAC peaks;
2. fixed 5-kb ATAC bins;
3. chromVAR motif-deviation scores; and
4. Signac ATAC gene-activity scores.

The four pure-guide negative controls are also included for 5-kb bins, chromVAR, and gene activity. Peak-level controls are not included because the peak-preparation helper defines ligand reference cells from `target_call`; the all-ligand peak batch therefore contains the 15 biological ligand perturbations.

Each GP input contains distance-bin midpoints (`X`), pseudobulk feature means (`means`), standard errors (`sems`), and calibrated fixed-noise variances (`mean_sem_sq`). Called peaks use six 50-µm bins; the other modalities use twelve 25-µm bins. Count-based features are retained when detected in strictly more than 5% of the eligible binned cells.


## Statistical test

For every feature within every perturbation, a zero-mean GP with a constant (Bias) covariance kernel is compared with a zero-mean GP with an RBF covariance kernel. Gaussian observation noise is fixed at the mean squared pseudobulk SEM across distance bins for that feature, $\sigma_f^2 = B^{-1}\sum_b SEM_{bf}^2$, rather than optimized.

The RBF model is optimized from three deterministic initial lengthscales—10, 50, and 200 µm—and the fit with the largest likelihood is retained. Each fit is allowed up to 1,000 L-BFGS-B iterations. The primary p-value uses the approximate boundary reference $0.5\chi_0^2 + 0.5\chi_1^2$, because the constant model is approached as the RBF lengthscale tends to infinity. Benjamini-Hochberg correction is performed **separately within each perturbation-by-modality screen** at 5% FDR. Ordinary $\chi_1^2$ and Benjamini-Yekutieli values are retained as sensitivity analyses.

The primary screens are unconstrained. Set `MIN_LENGTHSCALE_UM = 25.0` to reproduce the prespecified constrained sensitivity analysis for the 25-µm inputs; keep it separate from the primary results.


## 1. Project paths and run controls

The default settings inspect every available input and completed production result without launching long-running jobs. For a complete rerun, first set `PREPARE_MISSING_PEAK_INPUTS = True` to create normalized peak inputs for the other 14 ligands, then set `RUN_SCREENS = True`. New results are written beneath `results/notebook_multimodal_atac_gp_all_conditions/`; existing production outputs are never overwritten.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd


def find_workspace(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        runner = candidate / "gpclust_spatial" / "run_spatial_lrt.py"
        if runner.is_file() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find a parent containing gpclust_spatial/run_spatial_lrt.py and results/."
    )


WORKSPACE = find_workspace()
REPOSITORY = WORKSPACE / "perturb-tags"
GP_RUNNER = WORKSPACE / "gpclust_spatial" / "run_spatial_lrt.py"
GP_PYTHON = WORKSPACE / "gpclust_spatial" / ".venv_gpclust" / "Scripts" / "python.exe"
PEAK_PREP_SCRIPT = REPOSITORY / "prepare_atac_peak_distance_input.py"
SOURCE_H5MU = WORKSPACE / "all_lanes_merged_CATATAC_QC_RNA_processed_EB_manual_clustering_cell_types.h5mu"

FDR = 0.05
WORKERS_PER_SCREEN = 4
MAX_CONCURRENT_SCREENS = 2
CHUNK_SIZE = 25
MAX_OPTIMIZER_ITERS = 1000
RBF_START_LENGTHSCALES_UM = (10.0, 50.0, 200.0)
MIN_LENGTHSCALE_UM = None
DIAGNOSTIC_MAX_FEATURES = None  # Use an integer only for a smoke test.

INCLUDE_NEGATIVE_CONTROLS = True
PREPARE_MISSING_PEAK_INPUTS = False
RUN_SCREENS = False
OUTPUT_ROOT = WORKSPACE / "results" / "notebook_multimodal_atac_gp_all_conditions"
PEAK_BATCH_ROOT = WORKSPACE / "results" / "atac_peak_all_ligands_gp_assigned_50um"

assert GP_RUNNER.is_file(), GP_RUNNER
assert GP_PYTHON.is_file(), GP_PYTHON
print(f"Workspace: {WORKSPACE}")
print(f"GP interpreter: {GP_PYTHON}")


## 2. Register all ligand and control datasets

The `condition_id` names the 5-kb-bin and gene-activity directory. `chromvar_dataset` names the corresponding chromVAR directory. Keeping this mapping explicit prevents the negative-control aliases (`negative_control_1` versus `NTC_1`, for example) from being mixed up.


In [ ]:
LIGANDS = (
    "WNT5A", "WNT1", "BMP5", "BMP7", "JAG1",
    "DLL3", "VEGFB", "PDGFC", "EFNA3", "RSPO3",
    "SEMA6D", "NECTIN2", "NECTIN3", "ALB", "B2M",
)
NEGATIVE_CONTROLS = (
    ("NTC_1", "negative_control_1", "Negative control-1"),
    ("NTC_2", "negative_control_2", "Negative control-2"),
    ("NTC_3", "negative_control_3", "Negative control-3"),
    ("NTC_5", "negative_control_5", "Negative control-5"),
)

DATASETS = [
    {
        "dataset": ligand,
        "condition_id": ligand.lower(),
        "condition_label": ligand,
        "chromvar_dataset": ligand,
        "dataset_type": "ligand",
    }
    for ligand in LIGANDS
]
if INCLUDE_NEGATIVE_CONTROLS:
    DATASETS.extend(
        {
            "dataset": dataset,
            "condition_id": condition_id,
            "condition_label": condition_label,
            "chromvar_dataset": dataset,
            "dataset_type": "negative_control",
        }
        for dataset, condition_id, condition_label in NEGATIVE_CONTROLS
    )

dataset_table = pd.DataFrame(DATASETS)
dataset_table


## 3. Prepare missing all-ligand peak inputs

The 5-kb-bin, chromVAR, and gene-activity GP inputs already exist for all 19 datasets. The normalized called-peak input currently exists only for WNT5A. The code below reproduces the same peak workflow for every other ligand: assigned non-target cells are binned by within-embryoid-body distance to target-knockdown cells, peaks detected in more than 5% of binned cells are retained, raw counts are aggregated into six 50-µm pseudobulks, and the pseudobulks are transformed as `log1p(10000 * peak_count / total_ATAC_count)`. SEMs use the ratio-estimator delta method with count/library-size covariance.

Peak preparation is intentionally opt-in because it rereads the large H5Mu file for each ligand. It uses the validated workspace helper `prepare_atac_peak_distance_input.py` to calculate distances and raw moments; the log-library normalization is performed directly below.


In [ ]:
LEGACY_WNT5A_PEAK_INPUT = (
    WORKSPACE / "results" / "wnt5a_atac_peak_pseudobulk_logcpm_gp_assigned_50um"
    / "wnt5a_atac_peak_pseudobulk_logcpm_inputs.npz"
)


def peak_raw_input(dataset):
    slug = dataset["condition_id"]
    return PEAK_BATCH_ROOT / "raw_moments" / slug / f"{slug}_atac_peak_tfidf_inputs.npz"


def peak_batch_input(dataset):
    return PEAK_BATCH_ROOT / "conditions" / dataset["condition_id"] / "gp_inputs.npz"


def peak_input_path(dataset):
    batch_input = peak_batch_input(dataset)
    if batch_input.is_file():
        return batch_input
    if dataset["dataset"] == "WNT5A" and LEGACY_WNT5A_PEAK_INPUT.is_file():
        return LEGACY_WNT5A_PEAK_INPUT
    return batch_input


def build_peak_moment_command(dataset):
    raw_dir = peak_raw_input(dataset).parent
    return [
        sys.executable, str(PEAK_PREP_SCRIPT),
        "--input", str(SOURCE_H5MU),
        "--output", str(raw_dir),
        "--target", dataset["condition_label"],
        "--query-selection-mode", "assigned_non_target",
        "--detection-threshold", "0.05",
        "--bin-width-um", "50",
        "--max-distance-um", "300",
        "--normalization-order", "bin_then_tfidf",
        "--pseudobulk-idf-scope", "global_cells",
    ]


def derive_peak_logcpm_input(source_input, output_input, scale_factor=10_000.0):
    if output_input.is_file():
        return output_input
    with np.load(source_input, allow_pickle=False) as archive:
        required = {
            "X", "bin_counts", "pseudobulk_raw_count_sums",
            "pseudobulk_raw_count_squared_sums",
            "pseudobulk_raw_count_library_cross_sums",
            "pseudobulk_raw_library_sums",
            "pseudobulk_raw_library_squared_sums",
        }
        missing = required.difference(archive.files)
        if missing:
            raise ValueError(f"Peak raw-moment input is missing {sorted(missing)}")
        payload = {name: np.asarray(archive[name]) for name in archive.files}

    n = np.asarray(payload["bin_counts"], dtype=float)[:, None]
    count_sum = np.asarray(payload["pseudobulk_raw_count_sums"], dtype=float)
    count_sq_sum = np.asarray(payload["pseudobulk_raw_count_squared_sums"], dtype=float)
    count_library_sum = np.asarray(payload["pseudobulk_raw_count_library_cross_sums"], dtype=float)
    library_sum = np.asarray(payload["pseudobulk_raw_library_sums"], dtype=float)[:, None]
    library_sq_sum = np.asarray(payload["pseudobulk_raw_library_squared_sums"], dtype=float)[:, None]
    if np.any(n <= 1) or np.any(library_sum <= 0):
        raise ValueError("Every peak distance bin needs at least two cells and fragments.")

    ratio = count_sum / library_sum
    scaled = scale_factor * ratio
    means = np.log1p(scaled)
    residual_ss = np.maximum(
        count_sq_sum - 2.0 * ratio * count_library_sum + np.square(ratio) * library_sq_sum,
        0.0,
    )
    ratio_variance = n / (n - 1.0) * residual_ss / np.square(library_sum)
    sems = scale_factor / (1.0 + scaled) * np.sqrt(ratio_variance)
    mean_sem_sq = np.mean(np.square(sems), axis=0)
    valid = (
        np.all(np.isfinite(means), axis=0)
        & np.all(np.isfinite(sems), axis=0)
        & np.all(sems >= 0, axis=0)
        & np.isfinite(mean_sem_sq)
        & (mean_sem_sq > 0)
    )
    if not np.all(valid):
        raise ValueError(f"{int((~valid).sum())} peaks have invalid normalized summaries.")

    payload.update(
        means=np.ascontiguousarray(means),
        sems=np.ascontiguousarray(sems),
        mean_sem_sq=np.ascontiguousarray(mean_sem_sq),
        normalization_method=np.asarray("raw distance-bin pseudobulk, library-size normalize, log1p"),
        normalization_formula=np.asarray("log1p(10000 * C_bp / T_b)"),
        normalization_order=np.asarray("bin_then_log_cpm"),
        normalization_scale_factor=np.float64(scale_factor),
        idf_applied=np.asarray(False),
    )
    output_input.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output_input, **payload)
    return output_input


def prepare_peak_input(dataset):
    destination = peak_batch_input(dataset)
    if destination.is_file():
        return destination
    if not PEAK_PREP_SCRIPT.is_file():
        raise FileNotFoundError(PEAK_PREP_SCRIPT)
    if not SOURCE_H5MU.is_file():
        raise FileNotFoundError(SOURCE_H5MU)
    raw_input = peak_raw_input(dataset)
    if not raw_input.is_file():
        subprocess.run(build_peak_moment_command(dataset), cwd=REPOSITORY, check=True)
    return derive_peak_logcpm_input(raw_input, destination)


if PREPARE_MISSING_PEAK_INPUTS:
    for ligand_dataset in (row for row in DATASETS if row["dataset_type"] == "ligand"):
        existing_input = peak_input_path(ligand_dataset)
        if existing_input.is_file():
            print(f"Reusing called-peak input for {ligand_dataset['dataset']}: {existing_input}")
            continue
        print(f"Preparing called-peak input for {ligand_dataset['dataset']}")
        prepare_peak_input(ligand_dataset)
else:
    print("Peak preparation is disabled; existing peak inputs will be inventoried below.")


## 4. Build the perturbation-by-modality task matrix

This creates 60 ligand screens (15 ligands × 4 modalities) plus 12 negative-control screens (4 controls × 3 modalities). The generic GP runner calls every tested feature `gene` in its output table, even when the feature is a peak, genomic bin, or motif.


In [ ]:
def modality_specs(dataset):
    condition_id = dataset["condition_id"]
    chromvar_dataset = dataset["chromvar_dataset"]
    specs = {
        "ATAC 5-kb bins": {
            "input": WORKSPACE / "results" / "atac_ligand_distance_25um" / "snapatac2_tiles_5000bp" / "conditions" / condition_id / "gp_inputs.npz",
            "expected_bins": 12,
            "spatial_bin_width_um": 25,
            "reference_output": WORKSPACE / "results" / "atac_ligand_distance_25um" / "snapatac2_tiles_5000bp" / "conditions" / condition_id / "gp_lrt",
        },
        "chromVAR scores": {
            "input": WORKSPACE / "results" / "chromvar_all_ligands_gp_assigned_25um" / "datasets" / chromvar_dataset / f"{chromvar_dataset.lower()}_chromvar_inputs.npz",
            "expected_bins": 12,
            "spatial_bin_width_um": 25,
            "reference_output": WORKSPACE / "results" / "chromvar_all_ligands_gp_assigned_25um" / "datasets" / chromvar_dataset / "01_spatial_motif_screen",
        },
        "ATAC gene activity": {
            "input": WORKSPACE / "results" / "atac_ligand_distance_25um" / "signac_gene_activity" / "conditions" / condition_id / "gp_inputs.npz",
            "expected_bins": 12,
            "spatial_bin_width_um": 25,
            "reference_output": WORKSPACE / "results" / "atac_ligand_distance_25um" / "signac_gene_activity" / "conditions" / condition_id / "gp_lrt",
        },
    }
    if dataset["dataset_type"] == "ligand":
        specs = {
            "ATAC peaks": {
                "input": peak_input_path(dataset),
                "expected_bins": 6,
                "spatial_bin_width_um": 50,
                "reference_output": (
                    WORKSPACE / "results" / "wnt5a_atac_peak_pseudobulk_logcpm_gp_assigned_50um" / "01_spatial_peak_screen"
                    if dataset["dataset"] == "WNT5A" else None
                ),
            },
            **specs,
        }
    return specs


TASKS = []
for dataset in DATASETS:
    for modality, spec in modality_specs(dataset).items():
        TASKS.append({**dataset, "modality": modality, **spec})

task_table = pd.DataFrame(
    {
        "dataset": task["dataset"],
        "dataset_type": task["dataset_type"],
        "modality": task["modality"],
        "input": str(task["input"]),
    }
    for task in TASKS
)
task_table.groupby(["dataset_type", "modality"]).size().rename("screens").to_frame()


In [ ]:
REQUIRED_ARRAYS = {"X", "means", "sems", "mean_sem_sq", "gene_names", "gene_indices"}


def inspect_gp_input(task):
    path = task["input"]
    base = {"dataset": task["dataset"], "dataset_type": task["dataset_type"], "modality": task["modality"]}
    if not path.is_file():
        return {**base, "status": "missing", "features": np.nan, "distance_bins": task["expected_bins"], "input": str(path)}

    with np.load(path, allow_pickle=False) as data:
        missing = REQUIRED_ARRAYS.difference(data.files)
        if missing:
            raise ValueError(f"{task['dataset']} / {task['modality']} is missing {sorted(missing)}")
        X, means, sems = data["X"], data["means"], data["sems"]
        fixed_noise = data["mean_sem_sq"]
        feature_ids = data["gene_names"]
        if X.shape != (task["expected_bins"], 1):
            raise ValueError(f"Unexpected X shape for {task['dataset']} / {task['modality']}: {X.shape}")
        if means.shape != sems.shape or means.shape != (X.shape[0], feature_ids.size):
            raise ValueError(f"Incompatible feature arrays for {task['dataset']} / {task['modality']}")
        if fixed_noise.shape != (feature_ids.size,):
            raise ValueError(f"Incompatible fixed-noise array for {task['dataset']} / {task['modality']}")
        if not np.allclose(fixed_noise, np.mean(np.square(sems), axis=0), rtol=1e-8, atol=1e-12):
            raise ValueError(f"SEM calibration mismatch for {task['dataset']} / {task['modality']}")
        detection_min = float(np.min(data["detection"])) if "detection" in data.files else np.nan
        return {
            **base, "status": "ready", "features": feature_ids.size,
            "distance_bins": X.shape[0], "midpoint_min_um": float(X[0, 0]),
            "midpoint_max_um": float(X[-1, 0]), "minimum_detection": detection_min,
            "input": str(path),
        }


input_inventory = pd.DataFrame(inspect_gp_input(task) for task in TASKS)
input_inventory.groupby(["dataset_type", "modality", "status"]).size().rename("screens").to_frame()


### Processed-value interpretation

For called peaks and 5-kb bins, fragments are summed within each spatial pseudobulk, library-size normalized, and transformed as `log1p(10000 * feature_count / total_count)`. Signac gene-activity counts use the analogous gene-activity library total. Their SEMs propagate cell-level count and library-size variation while retaining count/library-size covariance. chromVAR inputs instead contain the mean and SEM of per-cell motif deviation scores in each distance bin. The GP screen does not renormalize these values.


## 5. Build and execute all GP screens

The command matrix below contains one independent GP screen per perturbation and modality. Two screens run concurrently by default, with four feature-fitting workers in each. Adjust those values to fit the machine. A first-N diagnostic subset is useful for checking paths and dependencies, but its q-values must not be reported because it changes the multiple-testing universe.


In [ ]:
def output_directory(task):
    modality_slug = task["modality"].lower().replace(" ", "_").replace("-", "")
    subset = "full" if DIAGNOSTIC_MAX_FEATURES is None else f"first_{DIAGNOSTIC_MAX_FEATURES}"
    constraint = "unconstrained" if MIN_LENGTHSCALE_UM is None else f"min_l{MIN_LENGTHSCALE_UM:g}um"
    return OUTPUT_ROOT / task["dataset"] / modality_slug / f"gp_lrt_{subset}_{constraint}"


def build_gp_command(task):
    command = [
        str(GP_PYTHON), str(GP_RUNNER),
        "--input", str(task["input"]),
        "--output", str(output_directory(task)),
        "--fdr", str(FDR),
        "--rbf-start-lengthscales", *map(str, RBF_START_LENGTHSCALES_UM),
        "--max-optimizer-iters", str(MAX_OPTIMIZER_ITERS),
        "--workers", str(WORKERS_PER_SCREEN),
        "--parallel-backend", "process",
        "--chunk-size", str(CHUNK_SIZE),
        "--expected-bins", str(task["expected_bins"]),
    ]
    if MIN_LENGTHSCALE_UM is not None:
        command.extend(["--rbf-lengthscale-minimum", str(MIN_LENGTHSCALE_UM)])
    if DIAGNOSTIC_MAX_FEATURES is not None:
        command.extend(["--diagnostic-max-genes", str(DIAGNOSTIC_MAX_FEATURES)])
    return command


def run_gp_screen(task):
    if not task["input"].is_file():
        raise FileNotFoundError(f"Missing input for {task['dataset']} / {task['modality']}: {task['input']}")
    output = output_directory(task)
    config_path = output / "run_config.json"
    if config_path.is_file():
        config = json.loads(config_path.read_text(encoding="utf-8"))
        if config.get("status") == "completed":
            return f"reused: {task['dataset']} / {task['modality']}"
    if output.exists() and any(output.iterdir()):
        raise FileExistsError(f"Refusing to overwrite incomplete/nonempty output: {output}")
    subprocess.run(build_gp_command(task), cwd=WORKSPACE, check=True)
    return f"completed: {task['dataset']} / {task['modality']}"


command_table = pd.DataFrame(
    {
        "dataset": task["dataset"], "modality": task["modality"],
        "input_ready": task["input"].is_file(),
        "command": subprocess.list2cmdline(build_gp_command(task)),
    }
    for task in TASKS
)

if RUN_SCREENS:
    missing_tasks = [task for task in TASKS if not task["input"].is_file()]
    if missing_tasks:
        labels = ", ".join(f"{task['dataset']}/{task['modality']}" for task in missing_tasks)
        raise FileNotFoundError(
            f"{len(missing_tasks)} GP inputs are missing: {labels}. "
            "Set PREPARE_MISSING_PEAK_INPUTS=True before the full run."
        )
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_SCREENS) as executor:
        futures = {executor.submit(run_gp_screen, task): task for task in TASKS}
        for future in as_completed(futures):
            print(future.result(), flush=True)
else:
    print("RUN_SCREENS is False; the all-condition command matrix was built but not executed.")

command_table.groupby(["modality", "input_ready"]).size().rename("screens").to_frame()


## 6. Aggregate completed results across all conditions

If `RUN_SCREENS` is false, this section reads the existing production results. Those outputs cover every ligand and negative control for 5-kb bins, chromVAR, and gene activity, plus the existing WNT5A peak screen. After a full notebook rerun, it instead reads the newly generated all-condition output tree.


In [ ]:
def available_result_directory(task):
    if RUN_SCREENS:
        candidate = output_directory(task)
    else:
        candidate = task["reference_output"]
    if candidate is None:
        return None
    table_path = candidate / "spatial_lrt_results.tsv"
    config_path = candidate / "run_config.json"
    return candidate if table_path.is_file() and config_path.is_file() else None


def load_results(task):
    directory = available_result_directory(task)
    if directory is None:
        return None
    table = pd.read_csv(directory / "spatial_lrt_results.tsv", sep="\t")
    table.insert(0, "dataset", task["dataset"])
    table.insert(1, "dataset_type", task["dataset_type"])
    table.insert(2, "modality", task["modality"])
    table.insert(4, "display_feature", table["gene"])
    if task["modality"] == "chromVAR scores":
        with np.load(task["input"], allow_pickle=False) as data:
            motif_names = dict(zip(data["motif_ids"].astype(str), data["motif_names"].astype(str)))
        table["display_feature"] = table["gene"].map(motif_names).fillna(table["gene"])
    return table


RESULTS = {}
for task in TASKS:
    table = load_results(task)
    if table is not None:
        RESULTS[(task["dataset"], task["modality"])] = table

run_summary = pd.DataFrame(
    {
        "dataset": dataset, "modality": modality,
        "tested": len(table),
        "fit_failures": int((~table["fit_success"]).sum()),
        "significant_at_5pct_fdr": int(table["significant_fdr"].sum()),
        "minimum_q_value": float(table["fdr_q_value"].min()),
    }
    for (dataset, modality), table in RESULTS.items()
)
print(f"Loaded {len(run_summary)} completed perturbation-by-modality screens.")
run_summary.sort_values(["modality", "dataset"]).reset_index(drop=True)


In [ ]:
TOP_COLUMNS = [
    "dataset", "dataset_type", "modality", "gene", "display_feature",
    "detection_fraction", "expression_range", "likelihood_ratio_statistic",
    "p_value", "fdr_q_value", "rbf_lengthscale_um", "significant_fdr",
]
top_hit_per_screen = pd.concat(
    [
        table.loc[table["fit_success"]]
        .sort_values(["p_value", "likelihood_ratio_statistic"], ascending=[True, False])
        .head(1)[TOP_COLUMNS]
        for table in RESULTS.values()
    ],
    ignore_index=True,
).sort_values(["modality", "fdr_q_value", "dataset"])
top_hit_per_screen.reset_index(drop=True)


## Reproducibility checklist

For the final all-ligand rerun, use `DIAGNOSTIC_MAX_FEATURES = None`, keep ligand/modality screens in separate output directories, and archive each `run_config.json` plus its input SHA-256 hash. Confirm that all 60 ligand screens completed, inspect optimizer failures and boundary flags, and analyze the negative controls alongside ligand results. If a minimum lengthscale is imposed, label that output as a constrained sensitivity analysis rather than combining it with the unconstrained primary screen.
